**IMPORTANTE**: Esta libreta lee una copia del conjunto de datos **Pasajeros de transporte público en Chicago**

El conjunto de datos está disponible para descargar en su sitio web oficial: https://data.cityofchicago.org/Transportation/CTA-Ridership-Daily-Boarding-Totals/6iiy-9s97

Así como en el repositorio GitHub que se ha habilitado para el curso.

In [ ]:
# Read the dataset into a pandas dataframe structure
url = "https://raw.githubusercontent.com/gakudo-ai/open-datasets/refs/heads/main/CTA_-_Ridership_-_Daily_Boarding_Totals.csv"

#...

Unos sencillos pasos de preprocesamiento. No necesitaremos la columna *total_rides*, que simplemente representa la suma de los viajes en autobús y tren por día. Además, sabemos que el conjunto de datos contiene duplicados, así que los eliminaremos.

Por último, y más importante, este es un conjunto de datos de **series temporales**, por lo que es fundamental asegurarnos de tener todos los datos ordenados -e incluso a ser posible, *indexados*- por fecha. Cada instancia está asociada a un día.

Vamos a visualizar un fragmento de nuestra preciosa serie temporal, con la ayuda de la librería Matplotlib: escogeremos el primer trimestre de 2020.

## Entrenando una RNN muy simple: ¡tan solo una capa de una neurona!

Este modelo tan simple considera una serie temporal univariante: los instantes de tiempo *t* tienen asociado un solo atributo: pasajeros en tren por día. Dividimos esta serie temporal en conjuntos de entrenamiento, validación y prueba.

Para una mejor interpretación, escalamos los datos de la serie temporal dividiendo los valores entre 10^6 para representar "millones de pasajeros por día".

Es necesario convertir estos DataFrames en una estructura específica de Keras para modelar y utilizar correctamente series temporales:

Para la forma de la capa de entrada, especificar:

* La longitud de la secuencia de entrada: cuántas entradas anteriores se utilizan para calcular la salida actual. *None* permite aceptar cualquier tamaño de secuencia de modo flexible.
* Número de atributos o variables a pronosticar: *1* para series temporales univariadas (p. ej., pronosticar solo viajes en autobús), o mayor que *1* para series temporales multivariadas (p. ej., pronosticar viajes en autobús y tren conjuntamente).

Como novedad, en este ejemplo añadiremos un mecanismo de *early stopping* al proceso de entrenamiento que irá monitorizando cómo evoluciona la función de pérdida, es decir, la métrica MAE (error absoluto medio) a lo largo de las rondas (epochs).

MAE, MSE (error cuadrático medio) y MAPE (error porcentual absoluto medio) son las métricas más habituales para evaluar modelos de pronóstico de series temporales. En este caso, utilizaremos MAE.

Con una arquitectura tan simple, es más que probable que el proceso de entrenamiento mencionado tome mucho menos de las 500 epochs especificadas, pudiendo efectuarse la parada temprana si la función de pérdida y el MAE se estabilizan.

Por último, evaluamos el error en los conjuntos de validación y prueba.

## Un modelo de pronóstico algo más complejo

2 capas, la primera de ellas posee 32 neuronas

In [ ]:
import matplotlib.pyplot as plt

# Get the training history data
loss = history2.history['loss']
val_loss = history2.history['val_loss']
mae = history2.history['mae']
val_mae = history2.history['val_mae']
epochs = range(1, len(loss) + 1)

# Create a single figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plotting Loss on the first subplot
ax1.plot(epochs, loss, 'b', label='Training loss')
ax1.plot(epochs, val_loss, 'r', label='Validation loss')
ax1.set_title('Training and Validation Loss')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss (Huber)')
ax1.legend()
ax1.grid(True)

# Plotting MAE on the second subplot
ax2.plot(epochs, mae, 'b', label='Training MAE')
ax2.plot(epochs, val_mae, 'r', label='Validation MAE')
ax2.set_title('Training and Validation MAE')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('MAE')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## Construyamos ahora una RNN profunda

Dos capas ocultas con neuronas recurrentes.

Ahora te estarás preguntando: ¿por qué el rendimiento en el conjunto de prueba es mucho peor que en el conjunto de validación?

Recordemos las instrucciones donde se realizó la división:

```
rail_train = rides["rail_boardings"]["2016-01":"2018-12"] / 1e6
rail_valid = rides["rail_boardings"]["2019-01":"2019-05"] / 1e6
rail_test = rides["rail_boardings"]["2019-06":] / 1e6
```
Hemos dividido los datos en tres ventanas temporales consecutivas. Los modelos se entrenaron con datos entre 2016 y 2018. El conjunto de validación abarca los cinco meses posteriores a 2018, por lo que pronosticar con este conjunto es como hacer predicciones a corto plazo; Mientras que el conjunto de pruebas abarca junio de 2019, pronosticar aquí es como hacer predicciones a largo plazo. Parece lógico que pronosticar el futuro muy lejano sea menos preciso que pronosticar el futuro cercano, más aún cuando buena parte de los datos posteriores a 2019 sufrieron enormes cambios debido al COVID.


**EJERCICIO PROPUESTO**

* Toca investigar y probar con diferentes "ventanas temporales" para particionar tu dataset en conjuntos de entrenamiento, validación, y pruebas. Opcionalmente, prueba a definir tu propia arquitectura de RNN y ajustar otros hiperparámetros a tu elección, por ejemplo el learning_rate u otros asociados al entrenamiento.

¿Conseguirás reducir un poco ese "abismo" entre el error cometido en el conjunto de validación y el del conjunto de prueba?